In [ ]:
# pip install tensorflow

In [ ]:
import os
import pandas as pd
import numpy as np  
import itertools
import numpy.typing as npt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from tensorflow.keras.models import Sequential, Model 
from tensorflow.keras.layers import Input, Dropout, Dense, LSTM, TimeDistributed, RepeatVector
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import datetime
from keras import regularizers

os.getcwd()
os.chdir("C:\\path\\to\\CNC\\folder\\")

In [ ]:
def ts_train_test_split(
    ts: npt.ArrayLike, training_size: int
) -> tuple[pd.DataFrame, pd.DataFrame] | tuple[np.ndarray, np.ndarray]:
    """
    Time series train test split. Performs a single split of the series.

    Parameters:
    ----------
    ts: array-like
        univariate time series data set

    training_size: int
        Size of the training set. The test set length

    Returns:
    -------
    Tuple[pd.DataFrame, pd.DataFrame] | Tuple[np.ndarray, np.ndarray]
        A tuple containing the training and test sets,
        either as DataFrames or NumPy arrays
    """
    if training_size >= len(ts):
        raise ValueError("training_size must be < length of series")

    if isinstance(ts, pd.DataFrame):
        return ts.iloc[:training_size], ts.iloc[training_size:]

    return np.asarray(ts[:training_size]), np.asarray(ts[training_size:])


# define the autoencoder network model
def autoencoder_model(X):
    inputs = Input(shape=(X.shape[1], X.shape[2]))
    L1 = LSTM(16, activation='relu', return_sequences=True, 
              kernel_regularizer= regularizers.l2(0.00))(inputs)
    L2 = LSTM(4, activation='relu', return_sequences=False)(L1)
    L3 = RepeatVector(X.shape[1])(L2)
    L4 = LSTM(4, activation='relu', return_sequences=True)(L3)
    L5 = LSTM(16, activation='relu', return_sequences=True)(L4)
    output = TimeDistributed(Dense(X.shape[2]))(L5)    
    model = Model(inputs=inputs, outputs=output)
    return model

In [ ]:
path = 'datasets_pseudo/Threshold/'
names = os.listdir(path)
date_str = datetime.datetime.now().strftime("%m%d%Y_%H%M%S")
dfs = {}

for name in names:
    machine, material, component, _ = name.replace('.csv', '').split('_')
    tempdf = pd.read_csv(f"{path}\\{name}", index_col=0) 
    print(f"Size of {name}: {tempdf.shape}")

    dfs[name.replace('.csv', '')] = tempdf

df = pd.concat(dfs.values(), ignore_index=True)


In [ ]:
columns_to_predict = ['CURRENT|1', 'CURRENT|2', 'CURRENT|3', 'CURRENT|6']

latent_dims = [4, 8, 10, 15]
contamination_values = [0.005, 0.001, 0.01, 0.05]
epochs_list = [25, 50, 75, 100, 150 ]
batch_sizes = [32, 64, 128]

feature_sets = {
    'all_sensors': list(set(df.columns) - set(['Machine', 'Material', 'Component']) - set(['CURRENT|1_Peak', 'CURRENT|2_Peak', 'CURRENT|3_Peak', 'CURRENT|6_Peak'])), 
    'torques' : ['TORQUE|1', 'TORQUE|2', 'TORQUE|3', 'TORQUE|6'], 
    'cmd_speed' : ['CMD_SPEED|1', 'CMD_SPEED|2', 'CMD_SPEED|3', 'CMD_SPEED|6']
}


# adjust as required. Was set as a constant since the computation takes a long time to run 
numeric_features = feature_sets['all_sensors']

In [ ]:
results = []


for name, target_column in list(itertools.product(names, columns_to_predict)):

    print(f'Training for column {target_column}')
    pseudo_label_col = f"{target_column}_Peak"

    axis = target_column[-2:]
    numeric_features_ = [x for x in numeric_features if axis in x]

    data = pd.read_csv(f"{path}\\{name}", index_col=0) 

    # divide to test and train
    train_size = round(data.shape[0]*0.7)
    target_train, target_test = ts_train_test_split(data[[target_column]], train_size)
    X_train, X_test = ts_train_test_split(data[numeric_features_], train_size)
    y_train, y_test = ts_train_test_split(data[[pseudo_label_col]], train_size)
    idx_train, idx_test = ts_train_test_split(data.index, train_size)

    # preprocess numerical variables
    scaler_X = MinMaxScaler()
    X_train = scaler_X.fit_transform(X_train)
    X_test = scaler_X.transform(X_test)

    # Scale target variable
    scaler_y = MinMaxScaler()
    y_train = scaler_y.fit_transform(y_train)
    y_test = scaler_y.transform(y_test)

    y_train_pseudo = y_train.astype(bool)  # binary labels
    y_test_pseudo = y_test.astype(bool)  # binary labels


    # --- Configuration ---
    for contamination, epochs, batch_size in list(itertools.product(contamination_values, epochs_list, batch_sizes)):
        
        print(f'training for contamination value = {contamination}, number of epochs = {epochs}, batch size = {batch_size}')

        # reshape inputs for LSTM [samples, timesteps, features]
        X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
        print("Training data shape:", X_train.shape)
        X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
        print("Test data shape:", X_test.shape)

        autoencoder = autoencoder_model(X_train)
        autoencoder.compile(optimizer='adam', loss='mae')
        autoencoder.summary()

        autoencoder.fit(X_train, X_train, epochs=epochs, batch_size=batch_size, validation_data=(X_test, X_test), verbose=0)

        # Reconstruction and anomaly detection
        X_recon = autoencoder.predict(X_test)
        X_pred = X_recon.reshape(X_recon.shape[0], X_recon.shape[2])
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[2])
        X_train = X_train.reshape(X_train.shape[0], X_train.shape[2])

        pred_errors = np.mean(np.abs(X_test - X_pred), axis=1)

        # Set threshold by contamination -> based on top percentiles
        threshold = np.quantile(pred_errors, 1 - contamination)
        peaks_detected = pred_errors > threshold

        # Evaluate
        precision = precision_score(y_test_pseudo, peaks_detected, zero_division=0)
        recall = recall_score(y_test_pseudo, peaks_detected, zero_division=0)
        f1 = f1_score(y_test_pseudo, peaks_detected, zero_division=0)
        n_detected = np.sum(peaks_detected)

        results.append({
            'Dataset': name, 
            'Column': target_column,
            'Contamination': contamination,
            'Epochs': epochs, 
            'Batch size': batch_size,
            'Precision': precision,
            'Recall': recall,
            'F1': f1,
            'Detected_Peaks': n_detected
        })

        print(f"Precision: {precision:.3f}")
        print(f"Recall:    {recall:.3f}")
        print(f"F1 Score:  {f1:.3f}")
       
        plt.figure(figsize=(14, 6))
        plt.plot(idx_test, target_test, label=f'{target_column} (Actual)', color='blue')
        plt.scatter(
            idx_test[peaks_detected],
            target_test.iloc[peaks_detected],
            color='red', label='Detected Peaks', marker='o'
        )
        plt.title(f"LSTM Autoencoder-based Peak Detection for {target_column}")
        plt.xlabel("Index")
        plt.ylabel("Current")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.sort_values(by='F1', ascending=False, inplace=True)

results_df.to_csv(f"Results/LSTM_results_{date_str}_all.csv")

